# Computer-Aided Drug Discovery: LSD1 (KDM1A) Inhibitors

**Target:** Lysine-specific histone demethylase 1A
**ChEMBL ID:** CHEMBL6136 | **UniProt:** O60341
**Disease context:** Acute myeloid leukaemia, small-cell lung cancer

LSD1 is a FAD-dependent histone demethylase that removes methyl marks from
histone H3 lysine 4. It is an epigenetic "eraser" and a validated oncology
target.

## Project scope

| Phase | Content |
|---|---|
| 1 | Data acquisition from ChEMBL |
| 2 | Molecular curation and activity classification |
| 3 | Physicochemical descriptor analysis |
| 4 | Fragment and scaffold analysis |
| 5 | Similarity screening (Enamine + drug repurposing) |
| 6 | Molecular docking and interaction analysis |

---

# Phase 1 - Data Acquisition

## 1.1 Environment setup

Installs dependencies and sets the working directory. Re-run at the start of
every Colab session; installed packages do not persist between sessions.

In [ ]:
!pip install -q rdkit chembl_webresource_client

import os

# Try to persist results to Google Drive; fall back to session storage.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = "/content/drive/MyDrive/CADD_LSD1"
except Exception as e:
    print("Drive unavailable (", type(e).__name__, ") - using session storage.")
    print("IMPORTANT: download your output files before closing this tab.")
    PROJECT = "/content/CADD_LSD1"

os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
print("Working directory:", os.getcwd())

## 1.2 Download bioactivity data

All IC50 records for CHEMBL6136 are retrieved from the ChEMBL web services
API. Records are fetched 1000 at a time; the guard prevents re-downloading
when the file already exists.

In [ ]:
import requests
import pandas as pd
import numpy as np

RAW = "LSD1_IC50_raw.csv"

if not os.path.exists(RAW):
    BASE = "https://www.ebi.ac.uk/chembl/api/data/activity"
    params = {
        "target_chembl_id": "CHEMBL6136",
        "standard_type": "IC50",
        "format": "json",
        "limit": 1000,
        "offset": 0,
    }
    rows = []
    while True:
        r = requests.get(BASE, params=params, timeout=120).json()
        rows.extend(r["activities"])
        print("fetched:", len(rows))
        if not r["page_meta"]["next"]:
            break
        params["offset"] += 1000

    keep = ["molecule_chembl_id", "canonical_smiles", "standard_value",
            "standard_units", "standard_relation", "pchembl_value",
            "assay_chembl_id", "assay_description", "assay_type",
            "document_chembl_id"]
    pd.DataFrame(rows)[keep].to_csv(RAW, index=False)
    print("Saved", RAW)
else:
    print("Using existing", RAW)

## 1.3 Inspect the raw dataset

Before any filtering, establish what the data contains: units, measurement
relations, missing values and compound count.

**Why this matters.** IC50 values recorded in different units cannot be
log-transformed together. Checking first prevents a silent error that would
propagate through the entire analysis.

In [ ]:
df = pd.read_csv(RAW)

print("RAW SHAPE:", df.shape)
print("\n--- UNITS ---")
print(df["standard_units"].value_counts(dropna=False))
print("\n--- RELATIONS ---")
print(df["standard_relation"].value_counts(dropna=False))
print("\n--- MISSING ---")
print(df[["canonical_smiles", "standard_value", "pchembl_value"]].isna().sum())
print("\nUnique compounds:", df["molecule_chembl_id"].nunique())

## 1.4 Filter to analysable records

| Filter | Reason |
|---|---|
| `units == nM` | Mixed units cannot be pooled |
| `relation == '='` | `<` and `>` are censored limits, not measurements |
| value and SMILES present | Required downstream |
| `value > 0` | Zero or negative values break the logarithm |

A funnel is printed so every discarded row is accounted for. Censored
records (`<`, `>`) make up a substantial fraction of the raw data; their
removal is deliberate and is reported in the methods.

In [ ]:
d = df.copy()
n = len(d)
print(f"start                 {n:>6}")

d = d[d["standard_units"] == "nM"]
print(f"units == nM           {len(d):>6}   (-{n-len(d)})"); n = len(d)

d = d[d["standard_relation"] == "="]
print(f"relation == '='       {len(d):>6}   (-{n-len(d)})"); n = len(d)

d = d.dropna(subset=["standard_value", "canonical_smiles"])
print(f"has value + SMILES    {len(d):>6}   (-{n-len(d)})"); n = len(d)

d = d[d["standard_value"] > 0]
print(f"value > 0             {len(d):>6}   (-{n-len(d)})")

## 1.5 Convert IC50 to pIC50

$$pIC_{50} = -\log_{10}(IC_{50}\ \text{in M}) = 9 - \log_{10}(IC_{50}\ \text{in nM})$$

Higher pIC50 means greater potency. The log scale makes the distribution
approximately symmetric and suitable for statistical testing.

**Validation.** Computed values are compared against ChEMBL's own
`pchembl_value` column - an independent check of both the formula and the
unit filter. Agreement to within rounding confirms the conversion.

In [ ]:
d = d.copy()
d["pIC50"] = 9 - np.log10(d["standard_value"])

chk = d.dropna(subset=["pchembl_value"])
diff = (chk["pIC50"] - chk["pchembl_value"]).abs()

print("Rows cross-checked:", len(chk))
print("Max difference:", round(diff.max(), 4))
print("Rows differing by >0.01:", (diff > 0.01).sum())
print("\npIC50 range:", round(d["pIC50"].min(), 2), "to", round(d["pIC50"].max(), 2))

---
# Phase 2 - Molecular Curation

## 2.1 Standardise structures

Each SMILES string is parsed, desalted, reduced to its largest fragment,
sanitised, and rewritten in canonical form.

**Why.** The same compound may appear as a free base, a hydrochloride salt,
or written with different but equivalent SMILES. Without canonicalisation,
identical structures would be counted as distinct compounds.

In [ ]:
from rdkit import Chem
from rdkit.Chem import SaltRemover
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

remover = SaltRemover.SaltRemover()

def clean_smiles(smi):
    """Parse, desalt, keep largest fragment, sanitize, canonicalise."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = remover.StripMol(mol)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) > 1:
        mol = max(frags, key=lambda m: m.GetNumAtoms())
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol)

d["clean_smiles"] = d["canonical_smiles"].apply(clean_smiles)
print("Failed to parse:", d["clean_smiles"].isna().sum())
d = d.dropna(subset=["clean_smiles"])
print("Remaining rows:", len(d))

## 2.2 Aggregate replicate measurements

Compounds are frequently tested in several assays and publications. Records
are grouped by canonical structure and summarised by the **median** pIC50,
which is robust to outlying measurements.

The standard deviation across replicates is retained as a reliability
indicator: compounds where independent laboratories disagree by more than
one log unit are flagged.

In [ ]:
agg = (
    d.groupby("clean_smiles")
     .agg(
         chembl_id = ("molecule_chembl_id", "first"),
         pIC50     = ("pIC50", "median"),
         pIC50_std = ("pIC50", "std"),
         n_measure = ("pIC50", "size"),
     )
     .reset_index()
)

print("Unique compounds:", len(agg))
print("\nMeasurements per compound:")
print(agg["n_measure"].describe())

noisy = agg[(agg["n_measure"] > 1) & (agg["pIC50_std"] > 1)]
print("\nHigh-disagreement compounds (std > 1 log unit):", len(noisy))

## 2.3 Activity classification

| Class | pIC50 | IC50 equivalent |
|---|---|---|
| Active | >= 7 | <= 100 nM |
| Moderate | 5 - 7 | 100 nM - 10 uM |
| Inactive | < 5 | > 10 uM |

These thresholds follow standard medicinal-chemistry convention. Unlike a
single 10 uM cutoff - which would place almost every compound in one class -
they produce populated groups suitable for comparative statistics.

In [ ]:
def classify(p):
    if p >= 7:
        return "Active"
    elif p >= 5:
        return "Moderate"
    else:
        return "Inactive"

agg["Class"] = agg["pIC50"].apply(classify)
ORDER = ["Active", "Moderate", "Inactive"]

print(agg["Class"].value_counts().reindex(ORDER).to_string())
print("\nTotal compounds:", len(agg))

agg.to_csv("LSD1_clean.csv", index=False)
print("\nSaved: LSD1_clean.csv")

---
# Phase 3 - Physicochemical Descriptors

**Question:** are highly active LSD1 inhibitors chemically distinct from
weakly active ones in their bulk physicochemical properties?

## 3.1 Calculate descriptors

| Descriptor | Meaning |
|---|---|
| MW | Molecular weight |
| LogP | Lipophilicity (octanol/water partitioning) |
| TPSA | Topological polar surface area |
| HBD / HBA | Hydrogen bond donors / acceptors |
| RotB | Rotatable bonds (conformational flexibility) |
| Rings | Ring count |

In [ ]:
from rdkit.Chem import Descriptors, Crippen, Lipinski

df = pd.read_csv("LSD1_clean.csv")

def calc(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return pd.Series([np.nan] * 7)
    return pd.Series([
        Descriptors.MolWt(mol),
        Crippen.MolLogP(mol),
        Descriptors.TPSA(mol),
        Lipinski.NumHDonors(mol),
        Lipinski.NumHAcceptors(mol),
        Lipinski.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
    ])

DESC = ["MW", "LogP", "TPSA", "HBD", "HBA", "RotB", "Rings"]
df[DESC] = df["clean_smiles"].apply(calc)
df = df.dropna(subset=DESC)

print("Compounds with descriptors:", len(df))
df.head()

## 3.2 Molecular size filtering

Initial inspection of this dataset revealed extreme outliers (MW up to
~3500 Da, LogP as low as -25). These correspond to **peptide-based
inhibitors** - unsurprising, since the natural substrate of LSD1 is the
histone H3 tail.

Peptides are legitimate LSD1 inhibitors but fall outside small-molecule
chemical space and are unsuitable for the docking protocol used in Phase 6.
They are therefore excluded, and the number removed is reported.

In [ ]:
print("BEFORE FILTERING")
print(df[["MW", "LogP"]].describe().round(2).to_string())

mask = df["MW"].between(150, 900) & df["LogP"].between(-5, 8)
removed = df[~mask]

print(f"\n\nRemoving {len(removed)} compounds ({100*len(removed)/len(df):.1f}%)")
if len(removed):
    print("\nProperties of removed compounds:")
    print(removed[["MW", "LogP"]].describe().round(2).to_string())
    print("\nBy class:")
    print(removed["Class"].value_counts().to_string())

df = df[mask].copy()
print(f"\n\nRETAINED: {len(df)} compounds")
print(df["Class"].value_counts().reindex(ORDER).to_string())

df.to_csv("LSD1_descriptors.csv", index=False)

## 3.3 Descriptor summary by activity class

Median values per class. This table is the numerical counterpart to the
plots that follow; dense scatter plots exaggerate apparent overlap, so the
medians are the record of reference.

In [ ]:
summary = df.groupby("Class")[DESC].median().round(2).reindex(ORDER)
print("MEDIAN DESCRIPTOR VALUES BY CLASS\n")
print(summary.to_string())

print("\n\nCOMPOUND COUNTS")
print(df["Class"].value_counts().reindex(ORDER).to_string())

## 3.4 Visualisation

Scatter plots relate pIC50 to molecular weight and lipophilicity; box plots
show the class distributions directly. Colour is supported by marker shape
so class identity is never carried by colour alone.

In [ ]:
import matplotlib.pyplot as plt

COLORS  = {"Active": "#2a78d6", "Moderate": "#eb6834", "Inactive": "#1baf7a"}
MARKERS = {"Active": "o",       "Moderate": "s",       "Inactive": "^"}
GRID, AXIS, INK, MUTED = "#e1e0d9", "#c3c2b7", "#0b0b0b", "#898781"

def style(ax):
    ax.set_facecolor("#fcfcfb")
    ax.grid(True, color=GRID, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), facecolor="#fcfcfb")
for ax, xvar, xlabel in zip(axes, ["MW", "LogP"],
                            ["Molecular Weight (Da)", "LogP (lipophilicity)"]):
    for cls in ORDER:
        sub = df[df["Class"] == cls]
        ax.scatter(sub[xvar], sub["pIC50"], s=11, alpha=0.40, linewidths=0,
                   c=COLORS[cls], marker=MARKERS[cls],
                   label=f"{cls} (n={len(sub)})", zorder=3)
    style(ax)
    ax.set_xlabel(xlabel, color=INK, fontsize=10)
    ax.set_ylabel("pIC50", color=INK, fontsize=10)
    ax.set_title(f"pIC50 vs {xvar}", color=INK, fontsize=12, loc="left")

leg = axes[0].legend(frameon=False, fontsize=9, markerscale=2.2, loc="lower right")
for h in leg.legend_handles:
    h.set_alpha(1.0)
plt.tight_layout()
plt.savefig("fig1_scatter.png", dpi=300, facecolor="#fcfcfb")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), facecolor="#fcfcfb")
for ax, var, label in zip(axes, ["MW", "LogP"], ["Molecular Weight (Da)", "LogP"]):
    data = [df.loc[df["Class"] == c, var] for c in ORDER]
    bp = ax.boxplot(data, patch_artist=True, notch=True, widths=0.55,
                    medianprops=dict(color=INK, linewidth=1.6),
                    flierprops=dict(marker=".", markersize=3,
                                    markerfacecolor=MUTED,
                                    markeredgecolor="none", alpha=0.4))
    for patch, cls in zip(bp["boxes"], ORDER):
        patch.set_facecolor(COLORS[cls])
        patch.set_alpha(0.65)
        patch.set_edgecolor(AXIS)
    ax.set_xticks(range(1, len(ORDER) + 1))
    ax.set_xticklabels(ORDER)
    style(ax)
    ax.set_ylabel(label, color=INK, fontsize=10)
    ax.set_title(f"{var} by activity class", color=INK, fontsize=12, loc="left")

plt.tight_layout()
plt.savefig("fig2_boxplots.png", dpi=300, facecolor="#fcfcfb")
plt.show()

## 3.5 Statistical testing with effect sizes

With large samples, trivially small differences become statistically
significant. A p-value indicates only that a difference exists, not that it
is chemically meaningful.

**Cliff's delta** is therefore reported alongside each p-value:

| abs(delta) | Interpretation |
|---|---|
| < 0.15 | negligible |
| 0.15 - 0.33 | small |
| 0.33 - 0.47 | medium |
| > 0.47 | large |

A descriptor is treated as meaningfully different only when it is both
statistically significant **and** has a non-negligible effect size.

In [ ]:
from scipy.stats import mannwhitneyu

act = df[df["Class"] == "Active"]
ina = df[df["Class"] == "Inactive"]
alpha = 0.05 / len(DESC)

def interpret(dlt):
    a = abs(dlt)
    if a < 0.15: return "negligible"
    if a < 0.33: return "small"
    if a < 0.47: return "medium"
    return "large"

rows = []
for v in DESC:
    u, p = mannwhitneyu(act[v], ina[v], alternative="two-sided")
    delta = (2 * u) / (len(act) * len(ina)) - 1
    rows.append({
        "Descriptor": v,
        "Act": round(act[v].median(), 2),
        "Inact": round(ina[v].median(), 2),
        "Diff": round(act[v].median() - ina[v].median(), 2),
        "p": f"{p:.1e}",
        "Cliff_d": round(delta, 3),
        "Effect": interpret(delta),
        "Meaningful": "YES" if (p < alpha and abs(delta) >= 0.15) else "no",
    })

res = pd.DataFrame(rows)
print(f"Active (n={len(act)}) vs Inactive (n={len(ina)})")
print(f"Bonferroni threshold: p < {alpha:.4f}\n")
print(res.to_string(index=False))
res.to_csv("table1_descriptor_stats.csv", index=False)

---
# Phase 4 - Fragment and Scaffold Analysis

Phase 3 asks whether bulk properties separate actives from inactives.
Phase 4 asks a sharper question: **which specific structural features are
associated with potency?**

## 4.1 Define functional group patterns

Functional groups are detected using SMARTS substructure patterns. The set
below covers common medicinal-chemistry fragments plus one of particular
relevance to this target.

**Cyclopropylamine** is included deliberately. Tranylcypromine-derived LSD1
inhibitors carry this group as a covalent warhead that forms an adduct with
the FAD cofactor. If the analysis is working, this fragment should emerge as
strongly associated with activity - a built-in positive control.

In [ ]:
FRAGMENTS = {
    "Cyclopropylamine": "[NX3][CX4H]1[CX4H2][CX4H2]1",   # TCP covalent warhead
    "Amide":            "[CX3](=[OX1])[NX3]",
    "Sulfonamide":      "[SX4](=[OX1])(=[OX1])[NX3]",
    "Urea":             "[NX3][CX3](=[OX1])[NX3]",
    "Ester":            "[CX3](=[OX1])[OX2H0][#6]",
    "CarboxylicAcid":   "[CX3](=[OX1])[OX2H1]",
    "Ketone":           "[#6][CX3](=[OX1])[#6]",
    "Ether":            "[OD2]([#6])[#6]",
    "Hydroxyl":         "[OX2H]",
    "PrimaryAmine":     "[NX3;H2][CX4]",
    "TertiaryAmine":    "[NX3]([#6])([#6])[#6]",
    "Nitrile":          "[NX1]#[CX2]",
    "Halogen":          "[F,Cl,Br,I]",
    "Benzene":          "c1ccccc1",
    "Pyridine":         "n1ccccc1",
    "Pyrimidine":       "c1cncnc1",
    "Imidazole":        "c1cnc[nH]1",
    "Triazole":         "c1nnc[nH]1",
    "Thiophene":        "c1ccsc1",
    "Furan":            "c1ccoc1",
    "Indole":           "c1ccc2[nH]ccc2c1",
    "Quinoline":        "c1ccc2ncccc2c1",
    "Piperidine":       "C1CCNCC1",
    "Piperazine":       "C1CNCCN1",
    "Pyrrolidine":      "C1CCNC1",
    "Morpholine":       "C1COCCN1",
}

patterns = {}
for name, sma in FRAGMENTS.items():
    p = Chem.MolFromSmarts(sma)
    if p is None:
        print("INVALID SMARTS:", name, sma)
    else:
        patterns[name] = p

print(f"{len(patterns)} valid SMARTS patterns compiled")

## 4.2 Build the fragment presence matrix

Each compound is tested for the presence of every fragment, producing a
boolean matrix of compounds x fragments.

In [ ]:
df = pd.read_csv("LSD1_descriptors.csv")
mols = [Chem.MolFromSmiles(s) for s in df["clean_smiles"]]
print("Molecules parsed:", sum(m is not None for m in mols), "/", len(mols))

for name, patt in patterns.items():
    df[f"frag_{name}"] = [
        bool(m.HasSubstructMatch(patt)) if m is not None else False
        for m in mols
    ]

frag_cols = [f"frag_{n}" for n in patterns]
print("Fragment columns added:", len(frag_cols))

## 4.3 Fragment frequency and enrichment

For each fragment, the percentage of compounds containing it is computed
within each activity class.

Two enrichment ratios are reported:

- **Active/Inactive** - values above 1 indicate the fragment is more common
  among potent compounds
- **Inactive/Active** - the inverse, as defined in the project workflow

Fragments present in fewer than 20 compounds are excluded; frequencies based
on very small counts are unstable.

In [ ]:
MIN_COUNT = 20
rows = []

for name in patterns:
    col = f"frag_{name}"
    total = int(df[col].sum())
    if total < MIN_COUNT:
        continue

    pct = {c: 100 * df.loc[df["Class"] == c, col].mean() for c in ORDER}
    a, i = pct["Active"], pct["Inactive"]

    rows.append({
        "Fragment": name,
        "N_total": total,
        "pct_Active": round(a, 1),
        "pct_Moderate": round(pct["Moderate"], 1),
        "pct_Inactive": round(i, 1),
        "Act/Inact": round(a / i, 2) if i > 0 else np.inf,
        "Inact/Act": round(i / a, 2) if a > 0 else np.inf,
    })

freq = pd.DataFrame(rows).sort_values("Act/Inact", ascending=False)
print("FRAGMENT FREQUENCY BY CLASS (%)\n")
print(freq.to_string(index=False))
freq.to_csv("table2_fragment_frequency.csv", index=False)

## 4.4 Activity-associated fragments

For each fragment, compounds are split into those containing it and those
without, and the two pIC50 distributions are compared.

$$\Delta pIC_{50} = \text{median}(\text{with fragment}) - \text{median}(\text{without fragment})$$

- Positive: the fragment is associated with increased potency
- Near zero: little apparent association
- Negative: the fragment is associated with reduced potency

Significance is assessed with the Mann-Whitney U test. Because many
fragments are tested simultaneously, p-values are corrected using the
**Benjamini-Hochberg** false discovery rate procedure, which is less
conservative than Bonferroni and more appropriate for exploratory screening.

Cliff's delta is again reported, since a significant p-value alone does not
establish that a difference matters.

In [ ]:
def benjamini_hochberg(pvals):
    """Return BH-adjusted p-values (same order as input)."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    # enforce monotonicity from the largest p downwards
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(ranked, 0, 1)
    return out

rows = []
for name in patterns:
    col = f"frag_{name}"
    with_f = df.loc[df[col], "pIC50"]
    without_f = df.loc[~df[col], "pIC50"]
    if len(with_f) < MIN_COUNT or len(without_f) < MIN_COUNT:
        continue

    u, p = mannwhitneyu(with_f, without_f, alternative="two-sided")
    delta = (2 * u) / (len(with_f) * len(without_f)) - 1

    rows.append({
        "Fragment": name,
        "N_with": len(with_f),
        "Median_with": round(with_f.median(), 2),
        "N_without": len(without_f),
        "Median_without": round(without_f.median(), 2),
        "dpIC50": round(with_f.median() - without_f.median(), 2),
        "p_raw": p,
        "Cliff_d": round(delta, 3),
        "Effect": interpret(delta),
    })

frag_stats = pd.DataFrame(rows)
frag_stats["p_adj"] = benjamini_hochberg(frag_stats["p_raw"])
frag_stats["Significant"] = np.where(
    (frag_stats["p_adj"] < 0.05) & (frag_stats["Cliff_d"].abs() >= 0.15),
    "YES", "no")

frag_stats = frag_stats.sort_values("dpIC50", ascending=False)
disp = frag_stats.copy()
disp["p_raw"] = disp["p_raw"].apply(lambda x: f"{x:.1e}")
disp["p_adj"] = disp["p_adj"].apply(lambda x: f"{x:.1e}")

print("ACTIVITY-ASSOCIATED FRAGMENTS (sorted by dpIC50)\n")
print(disp.to_string(index=False))
frag_stats.to_csv("table3_fragment_activity.csv", index=False)

print("\n\nFragments meeting both criteria (BH p<0.05 and |Cliff's d|>=0.15):")
print(frag_stats.loc[frag_stats["Significant"] == "YES", "Fragment"].tolist())

## 4.5 Distribution plots for the most informative fragments

Box plots of pIC50 with and without each of the six fragments showing the
largest absolute effect size.

In [ ]:
top = (frag_stats.reindex(frag_stats["Cliff_d"].abs()
                          .sort_values(ascending=False).index)
                 .head(6))

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5), facecolor="#fcfcfb")
for ax, (_, r) in zip(axes.ravel(), top.iterrows()):
    col = f"frag_{r['Fragment']}"
    data = [df.loc[~df[col], "pIC50"], df.loc[df[col], "pIC50"]]
    bp = ax.boxplot(data, patch_artist=True, notch=True, widths=0.55,
                    medianprops=dict(color=INK, linewidth=1.6),
                    flierprops=dict(marker=".", markersize=3,
                                    markerfacecolor=MUTED,
                                    markeredgecolor="none", alpha=0.35))
    for patch, c in zip(bp["boxes"], ["#1baf7a", "#2a78d6"]):
        patch.set_facecolor(c)
        patch.set_alpha(0.65)
        patch.set_edgecolor(AXIS)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"absent\n(n={r['N_without']})",
                        f"present\n(n={r['N_with']})"], fontsize=9)
    style(ax)
    ax.set_ylabel("pIC50", color=INK, fontsize=9)
    ax.set_title(f"{r['Fragment']}   dpIC50 = {r['dpIC50']:+.2f}",
                 color=INK, fontsize=11, loc="left")

plt.tight_layout()
plt.savefig("fig3_fragment_boxplots.png", dpi=300, facecolor="#fcfcfb")
plt.show()

## 4.6 Molecular context - scaffold analysis

A fragment does not act in isolation. For the fragment most strongly
associated with activity, the **Bemis-Murcko scaffolds** of the active
compounds containing it are extracted and the most frequent are depicted.

The Murcko scaffold is the ring system plus the linkers connecting rings,
with all side chains removed - the structural core of the molecule.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Draw

best = frag_stats.sort_values("Cliff_d", ascending=False).iloc[0]["Fragment"]
print("Fragment with the strongest positive association:", best)

sel = df[(df[f"frag_{best}"]) & (df["Class"] == "Active")].copy()
print("Active compounds containing it:", len(sel))

def murcko(smi):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(smi)
    except Exception:
        return None

sel["scaffold"] = sel["clean_smiles"].apply(murcko)
sel = sel.dropna(subset=["scaffold"])
sel = sel[sel["scaffold"] != ""]

counts = sel["scaffold"].value_counts().head(8)
print(f"\nDistinct scaffolds: {sel['scaffold'].nunique()}")
print("\nTop scaffolds by frequency:")
for s, c in counts.items():
    med = sel.loc[sel["scaffold"] == s, "pIC50"].median()
    print(f"  n={c:<4} median pIC50={med:.2f}   {s}")

scaf_mols, legends = [], []
for s, c in counts.items():
    m = Chem.MolFromSmiles(s)
    if m is not None:
        med = sel.loc[sel["scaffold"] == s, "pIC50"].median()
        scaf_mols.append(m)
        legends.append(f"n={c}, median pIC50={med:.2f}")

img = Draw.MolsToGridImage(scaf_mols, legends=legends, molsPerRow=4,
                           subImgSize=(300, 260), returnPNG=False)
img.save("fig4_scaffolds.png")
img

## 4.7 Save the annotated dataset

The fragment matrix is written out for use in Phase 5.

In [ ]:
df.to_csv("LSD1_fragments.csv", index=False)
print("Saved LSD1_fragments.csv  ", df.shape)

print("\nFiles produced:")
for f in sorted(os.listdir(".")):
    if f.endswith((".csv", ".png")):
        print("  ", f)

---
## Interpretation checklist

Before moving to Phase 5, be able to answer:

1. Which fragments are **positively** associated with potency, and does
   cyclopropylamine appear among them as expected?
2. Do any fragments show a large p-value effect but a **negligible** Cliff's
   delta? Those must not be reported as meaningful.
3. Are the active compounds dominated by **one scaffold** or spread across
   several? This determines whether the similarity search in Phase 5 should
   expect a single chemotype or multiple.

---

**Next: Phase 5** - Morgan fingerprint similarity analysis, Enamine
diversity-set screening, and the approved-drug repurposing screen.